In [1]:
import tensorflow as tf

import pandas as pd
import numpy as np
import os, pickle


from network import  NetCNN2D_CSP, build_functional_cnn2D
from mne.decoding import CSP
from dataset_NewEEG import filter_rawEEG

from config_NewEEG import Config
import dataset_NewEEG


from quantizeml.models import QuantizationParams, quantize


import requests



server_ip =  "http://192.168.216.53:8000"




csv_path = os.path.join("streaming_data_file", "Conscious_EEG_20241203_152326063.csv")
data_raw = pd.read_csv(csv_path, skiprows=2, delimiter=';')
data_raw = data_raw[['Fp1', 'Fp2', 'F3', 'Fz', 'F4', 'C1', 'Cz', 'C2', 'P3', 'Pz', 'P4', 'T3', 'T4']]

def get_data_epoch():
    return np.array(data_raw.loc[0:2*250 - 1]).transpose(1,0)



from keras.utils import get_custom_objects

get_custom_objects()['NetCNN2D_CSP'] = NetCNN2D_CSP



model_leftRight = tf.keras.models.load_model(os.path.join('saved_models', 'best_model_leftRight_ALL.tf'))
with open(os.path.join('saved_models', 'csp_leftRight_ALL.pkl'), 'rb') as f:
    csp_leftRight = pickle.load(f)


model_upDown = tf.keras.models.load_model(os.path.join('saved_models', 'best_model_upDown_ALL.tf'))
with open(os.path.join('saved_models', 'csp_upDown_ALL.pkl'), 'rb') as f:
    csp_upDown = pickle.load(f)



In [2]:
fmodel_leftRight = build_functional_cnn2D(n_classes=2, input_shape=(3, 500, 1))
fmodel_upDown = build_functional_cnn2D(n_classes=2, input_shape=(3, 500, 1))

fmodel_leftRight.set_weights(model_leftRight.get_weights())
fmodel_upDown.set_weights(model_upDown.get_weights())


config = Config()

config.data_path = "EEG_RAW_DATA"
config.test_session = 1
config.used_classes = [['CLeft'], ['CDown']]

X, y = dataset_NewEEG.session_dataset(config, f'I{config.test_session:02d}')
X, _ = dataset_NewEEG.slice_EEG_epoch(config, X, y)


X_copy = X.copy()

X_leftRight = csp_leftRight.transform(X)
X_leftRight = np.expand_dims(X_leftRight, 3)

X_upDown = csp_upDown.transform(X_copy)
X_upDown = np.expand_dims(X_upDown, 3)



Xq_leftRight = ((X_leftRight - X_leftRight.min()) / (X_leftRight.max() - X_leftRight.min()) * 255).astype(np.uint8)
Xq_upDown = ((X_upDown - X_upDown.min()) / (X_upDown.max() - X_upDown.min()) * 255).astype(np.uint8)

qparams = QuantizationParams(input_weight_bits=8, weight_bits=4, activation_bits=4, per_tensor_activations=True)

quantized_model_leftRight = quantize(fmodel_leftRight, qparams=qparams, samples=Xq_leftRight, num_samples=120, batch_size=16, epochs=5)
quantized_model_upDown = quantize(fmodel_upDown, qparams=qparams, samples=Xq_upDown, num_samples=120, batch_size=16, epochs=5)

quantized_model_leftRight.summary()

8/8 [==============================] - 0s 3ms/step
Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 3, 500, 1)]       0         
                                                                 
 conv2d_4 (QuantizedConv2D)  (None, 3, 72, 8)          272       
                                                                 
 re_lu_6 (QuantizedReLU)     (None, 3, 72, 8)          2         
                                                                 
 conv2d_5 (QuantizedConv2D)  (None, 3, 15, 16)         2704      
                                                                 
 re_lu_7 (QuantizedReLU)     (None, 3, 15, 16)         2         
                                                                 
 flatten_2 (QuantizedFlatte  (None, 720)               0         
 n)                                                              
          

In [3]:
import akida
from cnn2snn import convert, set_akida_version, AkidaVersion


#virtual_chip = akida.devices()[0] #akida.AKD1000()
#chip = akida.devices()[0]
#print(chip)

with set_akida_version(AkidaVersion.v1):
    model_akida_leftRight = convert(quantized_model_leftRight)
    model_akida_upDown = convert(quantized_model_upDown)
    


#model_akida_upDown.map(chip)
#model_akida_leftRight.map(chip)

y = model_akida_leftRight.forward(Xq_leftRight)
y = model_akida_upDown.forward(Xq_upDown)

In [ ]:
from Serial_class import serial_class
import threading
import time
import sys

if __name__ == '__main__':

    #Gestion port com
    serial_flux = serial_class()
    serial_flux.init_port()

    thread_a = threading.Thread(target=serial_flux.reception, name='ta')
    thread_a.start()

    time.sleep(2)
    cptEEG =0

    if not serial_flux.data:
        print('No serial flux , program is closing')
        serial_flux.terminate()
        sys.exit()


    EEGraw_stack = []

    try:
        while True:
            while not serial_flux.data_queue.empty():
                array_data_list , ts_data = serial_flux.data_queue.get()

                for i in range(len(array_data_list)):
                    ts_value = ts_data[i] + (i*4) # Obtient le timestamp associe
                    EEGraw_stack.append(array_data_list[i][:13])
                    cptEEG +=1




                if len(EEGraw_stack) > 500:
                    last_epoch_raw = EEGraw_stack[-500:]
                    X = np.array(last_epoch_raw)
                    X = X.transpose(1,0)




                    if len(EEGraw_stack) > 10000:
                        EEGraw_stack[:] = last_epoch_raw

                    

                    X = filter_rawEEG(X, 0.5, 35)
                    X = np.expand_dims(X, 0)


                    X_copy = X.copy()

                    X_leftRight = csp_leftRight.transform(X)
                    X_leftRight = np.expand_dims(X_leftRight, 3)

                    X_upDown = csp_leftRight.transform(X_copy)
                    X_upDown = np.expand_dims(X_upDown, 3)

                    # Linear Quantization
                    Xq_leftRight = ((X_leftRight - X_leftRight.min()) / (X_leftRight.max() - X_leftRight.min()) * 255).astype(np.uint8)
                    Xq_upDown = ((X_upDown - X_upDown.min()) / (X_upDown.max() - X_upDown.min()) * 255).astype(np.uint8)

                    out_lr = np.argmax(model_akida_leftRight.forward(Xq_leftRight))
                    out_ud = np.argmax(model_akida_upDown.forward(Xq_upDown))

                    direction = "left" if out_lr == 0 else "right"
                    print(direction)
                    requests.post(f"{server_ip}/push_direction", json={"direction": direction, "confidence": 1.0})
                    direction = "up" if out_lr == 0 else "down"
                    print(direction)
                    requests.post(f"{server_ip}/push_direction", json={"direction": direction, "confidence": 1.0})



            
            time.sleep(0.002)  # Ajouter un delai pour eviter d'occuper 100 du CPU
            if (cptEEG >250 * 120):
                serial_flux.terminate() # Fermeture de la connexion EEG
                thread_a.join()
                break

    except KeyboardInterrupt:
        # Arreter le thread proprement lors d'une interruption (Ctrl + C)
        serial_flux.terminate() # Fermeture de la connexion EEG
        thread_a.join()


Port: COM3
  Description : Silicon Labs CP210x USB to UART Bridge (COM3)
  Hardware ID : USB VID:PID=10C4:EA60 SER=0289339D LOCATION=1-2
  Vendor ID   : 4292
  Product ID  : 60000
  Manufacturer: Silicon Labs
  Serial Num  : 0289339D
  Location    : 1-2
  Product     : None

port : COM3
left
up
left
up
left
up
left
up
left
up
left
up
right
down
left
up
right
down
left
up
left
up
left
up
left
up
right
down
right
down
left
up
right
down
right
down
right
down
right
down
left
up
left
up
left
up
left
up
left
up
left
up
left
up
left
up
left
up
left
up
left
up
left
up
left
up
left
up
left
up
left
up
left
up
left
up
left
up
left
up
left
up
left
up
left
up
left
up
left
up
left
up
right
down
right
down
right
down
left
up
left
up
right
down
left
up
left
up
right
down
right
down
right
down
right
down
right
down
right
down
right
down
right
down
right
down
left
up
left
up
left
up
left
up
left
up
left
up
left
up
right
down
left
up
right
down
left
up
left
up
left
up
left
up
right
down
right
down
right

Exception in thread ta:
Traceback (most recent call last):
  File "C:\Users\Hammo\AppData\Local\Programs\Python\Python310\lib\threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "c:\Users\Hammo\OneDrive\Desktop\Neurobus\eh-venv\lib\site-packages\ipykernel\ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "C:\Users\Hammo\AppData\Local\Programs\Python\Python310\lib\threading.py", line 953, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\Hammo\OneDrive\Desktop\Neurobus\EnhancedHuman-CSP\Serial_class.py", line 118, in reception
    ligneData =np.array_split(self.adu_to_data(d, self.nb_elec), 4)# extract ligne data
  File "c:\Users\Hammo\OneDrive\Desktop\Neurobus\EnhancedHuman-CSP\Serial_class.py", line 203, in adu_to_data
    temp_adu = data[j]
IndexError: list index out of range


right
down
right
down
left
up
left
up
right
down
right
down
left
up
right
down
right
down
right
down
right
down
right
down
right
down
left
up
right
down
right
down
left
up
left
up
left
up
left
up
left
up
left
up
left
up
left
up
left
up
right
down
right
down
left
up
left
up
left
up
right
down
right
down
right
down
right
down
right
down
right
down
right
down
left
up
left
up
left
up
left
up
left
up
left
up
left
up
left
up
left
up
left
up
left
up
left
up
left
up
left
up
left
up
left
up
left
up
left
up
left
up
left
up
left
up
left
up
left
up
left
up
left
up
left
up
left
up
right
down
right
down
left
up
left
up
right
down
left
up
right
down
right
down
left
up
right
down
right
down
right
down
right
down
left
up
left
up
right
down
left
up
left
up
left
up
left
up
left
up
left
up
left
up
left
up
left
up
right
down
left
up
left
up
left
up
left
up
right
down
right
down
right
down
right
down
right
down
right
down
right
down
left
up
left
up
left
up
left
up
left
up
left
up
left
up
left
up
left
up
lef